# Programación orientada a objetos con Python
## Video 7. Polimorfismo y clases abstractas (`abc`)

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Comprender qué es el polimorfismo y cómo permite escribir algoritmos numéricos independientes de cada familia concreta de funciones.
2. Analizar el polimorfismo implícito de Python y por qué conviene formalizar contratos explícitos en jerarquías matemáticas.
3. Definir clases base abstractas con `ABC` y `@abstractmethod` del módulo estándar `abc`.
4. Implementar métodos plantilla reutilizables en la clase abstracta apoyándose únicamente en sus métodos abstractos.

## 1. Planteamiento del problema

**Definición.** Sea $f: \mathbb{R} \to \mathbb{R}$ una función real diferenciable con derivada $f'$.
1. La *iteración de Newton-Raphson* a partir de una estimación inicial $x_0 \in \mathbb{R}$ genera la sucesión
   $$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}.$$
2. La *regla del trapecio compuesta* en un intervalo $[a, b]$ con $N \ge 1$ subintervalos de paso $h = \frac{b - a}{N}$ y nodos $x_j = a + j\,h$ aproxima la integral definida mediante
   $$T_N(f; a, b) = h\left(\frac{f(a) + f(b)}{2} + \sum_{j=1}^{N-1} f(x_j)\right).$$

Observa que ni Newton-Raphson ni la regla del trapecio necesitan saber si $f$ es un polinomio, una función exponencial o una función trigonométrica: solo necesitan que $f$ sepa evaluarse en un punto y calcular su derivada.

**Problema: algoritmos numéricos universales sobre un contrato común.**

- **Entrada:** familias distintas de funciones diferenciables (`Polinomio`, `FuncionExponencial`, `FuncionSinusoidal`).
- **Salida:** una clase base abstracta `FuncionDiferenciable(ABC)` que exija `evaluar(self, x)` y `derivar(self)`, y proporcione automáticamente `raiz_newton` e `integral_trapecio` a todas las subclases.

## 2. Polimorfismo implícito y sus límites

En programación orientada a objetos, el **polimorfismo** es la capacidad de utilizar objetos de clases distintas a través de una misma interfaz. En Python, el polimorfismo funciona de manera implícita: si dos objetos tienen un método `evaluar(x)`, una misma función puede recorrer ambos sin preguntar su tipo.

Sin embargo, el polimorfismo puramente implícito tiene una debilidad: si alguien crea una clase olvidando implementar `derivar()`, Python no advierte el error al construir el objeto, sino más tarde, cuando el algoritmo numérico intenta invocar el método faltante.

In [ ]:
from abc import ABC, abstractmethod
from math import exp, pi, sin, sqrt


class RectaSimple:
    def __init__(self, pendiente, ordenada):
        self.pendiente = pendiente
        self.ordenada = ordenada

    def evaluar(self, x):
        return self.pendiente * x + self.ordenada

    def derivar(self):
        return RectaSimple(0.0, self.pendiente)


class CuadraticaIncompleta:
    def evaluar(self, x):
        return x ** 2 - 2


def tabla_polimorfica(funciones, puntos):
    return {
        type(funcion).__name__: [funcion.evaluar(x) for x in puntos]
        for funcion in funciones
    }


objetos = [RectaSimple(2, -1), CuadraticaIncompleta()]
print("Evaluación polimórfica en [0, 1, 2]:", tabla_polimorfica(objetos, [0, 1, 2]))

try:
    CuadraticaIncompleta().derivar()
except AttributeError as error:
    print("Fallo tardío por falta de contrato:", error)

## 3. Clases base abstractas con `ABC` y `@abstractmethod`

El módulo estándar `abc` (*Abstract Base Classes*) permite expresar el contrato de manera formal:
- Una clase que hereda de `ABC` representa un concepto abstracto que **no puede instanciarse directamente**.
- Todo método decorado con `@abstractmethod` declara una operación obligatoria que cualquier subclase concreta debe implementar antes de poder crear instancias.

In [ ]:
class FuncionDiferenciableBase(ABC):
    @abstractmethod
    def evaluar(self, x):
        pass

    @abstractmethod
    def derivar(self):
        pass


class SubclaseIncompleta(FuncionDiferenciableBase):
    def evaluar(self, x):
        return x ** 2


for intento in (FuncionDiferenciableBase, SubclaseIncompleta):
    try:
        intento()
    except TypeError as error:
        print(f"Al instanciar {intento.__name__}:", error)

## 4. Métodos plantilla sobre métodos abstractos

Una gran ventaja de `ABC` es que la clase abstracta puede incluir **métodos concretos** escritos únicamente en términos de sus propios métodos abstractos. Así, `FuncionDiferenciable` implementa una sola vez `evaluar_derivada`, `raiz_newton` e `integral_trapecio`, y cada subclase concreta los hereda listos para usarse.

In [ ]:
class FuncionDiferenciable(ABC):
    @abstractmethod
    def evaluar(self, x):
        pass

    @abstractmethod
    def derivar(self):
        pass

    def evaluar_derivada(self, x):
        return self.derivar().evaluar(x)

    def raiz_newton(self, x0, tolerancia=1e-10, iteraciones_limite=50):
        x = float(x0)
        derivada = self.derivar()
        for _ in range(iteraciones_limite):
            valor = self.evaluar(x)
            if abs(valor) < tolerancia:
                return x
            pendiente = derivada.evaluar(x)
            if pendiente == 0:
                raise ZeroDivisionError("La derivada se anuló durante la iteración de Newton-Raphson.")
            x = x - valor / pendiente
        raise RuntimeError("El método de Newton-Raphson no convergió en el límite indicado.")

    def integral_trapecio(self, a, b, subintervalos=1000):
        if subintervalos <= 0:
            raise ValueError("El número de subintervalos debe ser positivo.")
        paso = (b - a) / subintervalos
        suma = 0.5 * (self.evaluar(a) + self.evaluar(b))
        for j in range(1, subintervalos):
            suma += self.evaluar(a + j * paso)
        return paso * suma


class Polinomio(FuncionDiferenciable):
    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self._coeficientes = tuple(lista)

    @property
    def coeficientes(self):
        return self._coeficientes

    def evaluar(self, x):
        resultado = 0
        for coeficiente in reversed(self._coeficientes):
            resultado = resultado * x + coeficiente
        return resultado

    def derivar(self):
        if len(self._coeficientes) == 1:
            return Polinomio([0])
        return Polinomio(
            exponente * coeficiente
            for exponente, coeficiente in enumerate(self._coeficientes)
            if exponente > 0
        )


class FuncionExponencial(FuncionDiferenciable):
    def __init__(self, coeficiente, tasa):
        self._coeficiente = float(coeficiente)
        self._tasa = float(tasa)

    def evaluar(self, x):
        return self._coeficiente * exp(self._tasa * x)

    def derivar(self):
        return FuncionExponencial(self._coeficiente * self._tasa, self._tasa)


pol = Polinomio([-2, 0, 1])  # x^2 - 2
exponencial = FuncionExponencial(1.0, 1.0)  # e^x

raiz_dos = pol.raiz_newton(1.5)
integral_exp = exponencial.integral_trapecio(0.0, 1.0, subintervalos=1000)

print("Raíz de x^2 - 2 por Newton-Raphson:", raiz_dos, "| sqrt(2) =", sqrt(2))
print("Integral de e^x en [0, 1]:", round(integral_exp, 6), "| e - 1 =", round(exp(1) - 1, 6))

### Tabla comparativa de estrategias polimórficas

| Estrategia | Verificación de contrato | Reutilización de algoritmos comunes | Cuándo conviene usarla |
|---|---|---|---|
| Polimorfismo implícito (sin clase base) | Tardía, en tiempo de ejecución | Mediante funciones libres externas | Prototipos pequeños o compatibilidad con clases ajenas |
| Herencia de clase concreta | Parcial (puede heredarse una implementación vacía) | Directa desde la superclase | Cuando existe una implementación base real por defecto |
| Clase base abstracta (`ABC`) | Temprana, al intentar instanciar una subclase incompleta | Mediante métodos concretos plantilla sobre `@abstractmethod` | Familias matemáticas que comparten algoritmos universales |

## 5. Ejercicios

**Ejercicio 1 (`ElementoEspacioNormado(ABC)`).** Diseña una clase abstracta `ElementoEspacioNormado(ABC)` que formalice un espacio vectorial normado:
- **Entrada:** métodos abstractos `__add__(self, otro)`, `escalar(self, factor)` y `norma(self)` (más un método abstracto `restar(self, otro)` o `__sub__(self, otro)`).
- **Salida:** métodos concretos heredados automáticamente por toda subclase: `distancia(self, otro)` que calcule $\|self - otro\|$ y `es_unitario(self, tolerancia=1e-9)` que decida si $|\|self\| - 1| < \text{tolerancia}$. Implementa `VectorPlano` y `VectorEspacio` como subclases concretas y comprueba ambos métodos concretos.

**Ejercicio 2 (`FuncionSinusoidal(FuncionDiferenciable)`).** Implementa la clase concreta `FuncionSinusoidal(FuncionDiferenciable)` para funciones de la forma
$$f(x) = A\sin(\omega x + \varphi)$$
utilizando la identidad exacta $f'(x) = A\omega\sin\left(\omega x + \varphi + \frac{\pi}{2}\right)$:
- **Entrada:** amplitud $A$, frecuencia angular $\omega$ y fase $\varphi$ en `__init__`.
- **Salida:** aplica `integral_trapecio(0, pi)` a $f(x) = \sin(x)$ para verificar que el área bajo un arco de seno es aproximadamente $2.0$, y calcula $\pi$ mediante `raiz_newton(3.0)`.

**Ejercicio 3 (combinación polimórfica `SumaDeFunciones(FuncionDiferenciable)`).** Implementa una clase concreta `SumaDeFunciones(FuncionDiferenciable)` que reciba dos instancias `primera` y `segunda` de `FuncionDiferenciable`:
- **Entrada:** dos instancias cualesquiera de `FuncionDiferenciable`.
- **Salida:** `evaluar(x)` devuelve `primera.evaluar(x) + segunda.evaluar(x)`, y `derivar()` devuelve `SumaDeFunciones(primera.derivar(), segunda.derivar())`. Úsala para resolver la ecuación mixta $e^x + x - 2 = 0$ combinando `FuncionExponencial(1, 1)` y `Polinomio([-2, 1])`.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class ElementoEspacioNormado(ABC):
    @abstractmethod
    def __add__(self, otro):
        pass

    @abstractmethod
    def __sub__(self, otro):
        pass

    @abstractmethod
    def escalar(self, factor):
        pass

    @abstractmethod
    def norma(self):
        pass

    def distancia(self, otro):
        return (self - otro).norma()

    def es_unitario(self, tolerancia=1e-9):
        return abs(self.norma() - 1.0) < tolerancia


class VectorPlanoNormado(ElementoEspacioNormado):
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __add__(self, otro):
        return VectorPlanoNormado(self.x + otro.x, self.y + otro.y)

    def __sub__(self, otro):
        return VectorPlanoNormado(self.x - otro.x, self.y - otro.y)

    def escalar(self, factor):
        return VectorPlanoNormado(factor * self.x, factor * self.y)

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)


class VectorEspacioNormado(ElementoEspacioNormado):
    def __init__(self, x, y, z):
        self.x = float(x)
        self.y = float(y)
        self.z = float(z)

    def __add__(self, otro):
        return VectorEspacioNormado(self.x + otro.x, self.y + otro.y, self.z + otro.z)

    def __sub__(self, otro):
        return VectorEspacioNormado(self.x - otro.x, self.y - otro.y, self.z - otro.z)

    def escalar(self, factor):
        return VectorEspacioNormado(factor * self.x, factor * self.y, factor * self.z)

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2 + self.z ** 2)


a2 = VectorPlanoNormado(0.6, 0.8)
b2 = VectorPlanoNormado(0.0, 0.0)
c3 = VectorEspacioNormado(1.0, 2.0, 2.0)
print("Distancia entre a2 y b2:", a2.distancia(b2), "| ¿a2 es unitario?", a2.es_unitario())
print("Norma de c3:", c3.norma(), "| ¿c3 es unitario?", c3.es_unitario())

In [ ]:
# Ejercicio 2
class FuncionSinusoidal(FuncionDiferenciable):
    def __init__(self, amplitud=1.0, frecuencia=1.0, fase=0.0):
        self._amplitud = float(amplitud)
        self._frecuencia = float(frecuencia)
        self._fase = float(fase)

    def evaluar(self, x):
        return self._amplitud * sin(self._frecuencia * x + self._fase)

    def derivar(self):
        return FuncionSinusoidal(
            self._amplitud * self._frecuencia,
            self._frecuencia,
            self._fase + pi / 2.0,
        )


seno = FuncionSinusoidal(1.0, 1.0, 0.0)
print("Integral de sin(x) en [0, pi]:", round(seno.integral_trapecio(0.0, pi, subintervalos=1000), 6))
print("Aproximación de pi desde x0 = 3.0:", seno.raiz_newton(3.0))

In [ ]:
# Ejercicio 3
class SumaDeFunciones(FuncionDiferenciable):
    def __init__(self, primera, segunda):
        if not isinstance(primera, FuncionDiferenciable) or not isinstance(segunda, FuncionDiferenciable):
            raise TypeError("Ambos sumandos deben ser instancias de FuncionDiferenciable.")
        self._primera = primera
        self._segunda = segunda

    def evaluar(self, x):
        return self._primera.evaluar(x) + self._segunda.evaluar(x)

    def derivar(self):
        return SumaDeFunciones(self._primera.derivar(), self._segunda.derivar())


ecuacion = SumaDeFunciones(FuncionExponencial(1.0, 1.0), Polinomio([-2, 1]))
solucion = ecuacion.raiz_newton(0.5)
print("Raíz de e^x + x - 2 = 0:", round(solucion, 10))
print("Verificación f(solución):", round(ecuacion.evaluar(solucion), 12))

## 7. Resumen

- El **polimorfismo** permite escribir algoritmos numéricos universales que operan sobre cualquier objeto que respete una interfaz común.
- El módulo estándar `abc` proporciona `ABC` y `@abstractmethod` para convertir convenciones informales en contratos verificados al instanciar.
- Los métodos concretos definidos dentro de una clase abstracta pueden invocar a sus propios métodos abstractos para evitar duplicar algoritmos en cada subclase.
- En el video final compararemos la composición frente a la herencia y cerraremos la serie con nuestro proyecto integrador de aritmética exacta sobre los números racionales.